In [ ]:
WITH calculations AS (SELECT
	item_no,
	height,
	avg_height,
	operator,
	stddev_height,
	avg_height+3*(stddev_height/sqrt(5)) AS ucl,
	avg_height-3*(stddev_height/sqrt(5)) AS lcl
FROM (
	SELECT
	item_no,
	height,
	operator,
	AVG(height) OVER (PARTITION BY operator ORDER BY item_no ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) AS avg_height,
	STDDEV(height) OVER (PARTITION BY operator ORDER BY item_no ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) AS stddev_height
	FROM manufacturing_parts
) AS subquery),
data_required AS (
	SELECT
	mp.item_no,
	mp.operator,
	ROW_NUMBER() OVER(PARTITION BY mp.operator ORDER BY mp.item_no) AS row_number,
	mp.height,
	c.avg_height,
	c.stddev_height,
	c.ucl,
	c.lcl,
	CASE WHEN mp.height>c.ucl OR mp.height<c.lcl THEN TRUE ELSE FALSE END AS alert
FROM manufacturing_parts mp
JOIN calculations c
ON c.item_no=mp.item_no
)

SELECT
	operator,
	row_number,
	height,
	avg_height,
	stddev_height,
	ucl,
	lcl,
	alert
FROM data_required
WHERE row_number>4
ORDER BY item_no;